# Part 1: Introduction to Classification & Evaluation

**Objective:** Load the synthetic health data, train a Logistic Regression model, and evaluate its performance.

## 1. Setup

Import necessary libraries.

In [17]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.impute import SimpleImputer

## 2. Data Loading

Implement the `load_data` function to read the dataset.

In [18]:
def load_data(file_path):
    """
    Load the synthetic health data from a CSV file.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        DataFrame containing the data
    """
    # Load the CSV file using pandas
    try:
        data = pd.read_csv(file_path, parse_dates=['timestamp'])
    except FileNotFoundError:
        print("File not exist!")
        data = None
    
    return data

## 3. Data Preparation

Implement `prepare_data_part1` to select features, split data, and handle missing values.

In [19]:
def prepare_data_part1(df, test_size=0.2, random_state=42):
    """
    Prepare data for modeling: select features, split into train/test sets, handle missing values.
    
    Args:
        df: Input DataFrame
        test_size: Proportion of data for testing
        random_state: Random seed for reproducibility
        
    Returns:
        X_train, X_test, y_train, y_test
    """
    # 1. Select relevant features
    data_selected = df[['age', 'systolic_bp', 'diastolic_bp', 'glucose_level', 'bmi']]
    target = df['disease_outcome']

    # 2. Split data
    X_train, X_test, y_train, y_test = train_test_split(data_selected, target, test_size=test_size, random_state=random_state)

    # 3. Impute missing values using a single imputer instance
    imputer = SimpleImputer(strategy='mean')
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    # Convert back to DataFrame to retain column names
    X_train = pd.DataFrame(X_train, columns=data_selected.columns)
    X_test = pd.DataFrame(X_test, columns=data_selected.columns)

    return X_train, X_test, y_train, y_test


## 4. Model Training

Implement `train_logistic_regression`.

In [20]:
def train_logistic_regression(X_train, y_train):
    """
    Train a logistic regression model.
    
    Args:
        X_train: Training features
        y_train: Training target
        
    Returns:
        Trained logistic regression model
    """
    # YOUR CODE HERE
    # Initialize and train a LogisticRegression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    return model

## 5. Model Evaluation

Implement `calculate_evaluation_metrics` to assess the model's performance.

In [21]:
def calculate_evaluation_metrics(model, X_test, y_test):
    """
    Calculate classification evaluation metrics.
    
    Args:
        model: Trained model
        X_test: Test features
        y_test: Test target
        
    Returns:
        Dictionary containing accuracy, precision, recall, f1, auc, and confusion_matrix
    """
    # YOUR CODE HERE
    # 1. Generate predictions
    y_pred = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    # 2. Calculate metrics: accuracy, precision, recall, f1, auc
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_prob)
    # 3. Create confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    # 4. Return metrics in a dictionary
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': conf_matrix
    }

## 6. Save Results

Save the calculated metrics to a text file.

In [22]:
# Create results directory and save metrics
# YOUR CODE HERE
# 1. Create 'results' directory if it doesn't exist
# 2. Format metrics as strings
# 3. Write metrics to 'results/results_part1.txt'

## 7. Main Execution

Run the complete workflow.

In [23]:
# Main execution
if __name__ == "__main__":
    # 1. Load data
    data_file = 'data/synthetic_health_data.csv'
    df = load_data(data_file)
    
    # 2. Prepare data
    X_train, X_test, y_train, y_test = prepare_data_part1(df)
    
    # 3. Train model
    model = train_logistic_regression(X_train, y_train)
    
    # 4. Evaluate model
    metrics = calculate_evaluation_metrics(model, X_test, y_test)
    
    # 5. Print metrics
    for metric, value in metrics.items():
        if metric != 'confusion_matrix':
            print(f"{metric}: {value:.4f}")
    
    # 6. Save results
    # (Your code for saving results)
    results_dir = 'results'
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    metrics = calculate_evaluation_metrics(model, X_test, y_test)
    # Write metrics to 'results/results_part1.txt'
    with open(os.path.join(results_dir, 'results_part1.txt'), 'w') as f:
        for metric, value in metrics.items():
            if metric != 'confusion_matrix':
                if value is not None and isinstance(value, (int, float)):
                    f.write(f"{metric}: {value:.4f}\n")


    # 7. Interpret results
    interpretation = interpret_results(metrics)
    print("\nResults Interpretation:")
    for key, value in interpretation.items():
        print(f"{key}: {value}")

accuracy: 0.9168
precision: 0.6615
recall: 0.3007
f1: 0.4135
auc: 0.9084

Results Interpretation:
best_metric: accuracy
worst_metric: recall
imbalance_impact_score: 0.6720050782550783


## 8. Interpret Results

Implement a function to analyze the model performance on imbalanced data.

In [24]:
def interpret_results(metrics):
    """
    Analyze model performance on imbalanced data.
    
    Args:
        metrics: Dictionary containing evaluation metrics
        
    Returns:
        Dictionary with keys:
        - 'best_metric': Name of the metric that performed best
        - 'worst_metric': Name of the metric that performed worst
        - 'imbalance_impact_score': A score from 0-1 indicating how much
          the class imbalance affected results (0=no impact, 1=severe impact)
    """
    # Extracting metrics
    accuracy = metrics['accuracy']
    precision = metrics['precision']
    recall = metrics['recall']
    f1 = metrics['f1']
    auc = metrics['auc']
    
    # 1. Determine the best and worst metric based on the value
    metric_values = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }
    
    best_metric = max(metric_values, key=metric_values.get)
    worst_metric = min(metric_values, key=metric_values.get)
    
    # 2. Calculate imbalance impact score (simplified as the difference between accuracy and recall/F1)
    imbalance_impact_score = 0.0
    
    if recall < accuracy:
        imbalance_impact_score = 1 - (recall / accuracy)
    
    if f1 < accuracy:
        imbalance_impact_score = max(imbalance_impact_score, 1 - (f1 / accuracy))
        
    # 3. Return the results as a dictionary
    return {
        'best_metric': best_metric,
        'worst_metric': worst_metric,
        'imbalance_impact_score': imbalance_impact_score
    }
